In [2]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from PIL import ImageGrab
import serial
import time
import cv2
import win32gui
from fractions import Fraction
import requests

## Functions cell

In [3]:
#checks if the image is black on white background or not
def isBlackOnWhite(arr):
    if np.sum(arr==255) > np.sum(arr==0):
        return True
    else:
        return False
    
def decimate(arr, ratio):
    frac = Fraction(ratio).limit_denominator(1000)
    num = frac.numerator
    den = frac.denominator

    idArr = np.arange(0,len(arr))
    maskArr = np.zeros(len(arr), dtype=bool)
    maskArr[idArr%den < num] = True
    
    return arr[maskArr]

def getImage(name, k, thresh, mode, customWindowSize, bounds):
    #finds the designated window and outputs the coordinates of its diagonal corners
    hwnd = win32gui.FindWindow(None, name)
    left, top, right, bottom = win32gui.GetWindowRect(hwnd)

    #gets image of the selected portion of the screen, grayscales it and resizes it
    if customWindowSize == True:
        top, bottom, left, right = bounds
    
    img = ImageGrab.grab(bbox=(left, top, right, bottom)).convert("L")
    w, h = img.size
    imgResized = img.resize((int(w*k), int(h*k)))
    arr = np.array(imgResized)
    
    #applies a threshold to get a monochrome array
    if mode == "blackFrontWhiteBack":
        binaryArr = (arr < thresh).astype(np.uint8)
    elif mode == "blackBackWhiteFront":
        binaryArr = (arr > thresh).astype(np.uint8)
    elif mode == "dynamic":
        if isBlackOnWhite(arr) == True:
            binaryArr = (arr < thresh).astype(np.uint8)
        else:
            binaryArr = (arr > thresh).astype(np.uint8)
    else:
        binaryArr = (arr < thresh).astype(np.uint8)
        
    yLen, xLen = binaryArr.shape
    
    return binaryArr, xLen, yLen

def getImageFromFile(file, k, thresh, mode):
    #gets image of the selected portion of the screen, grayscales it and resizes it
    img = Image.open(file).convert("L")
    w, h = img.size
    imgResized = img.resize((int(w*k), int(h*k)))
    arr = np.array(imgResized)
    
    #applies a threshold to get a monochrome array
    if mode == "blackFrontWhiteBack":
        binaryArr = (arr < thresh).astype(np.uint8)
    elif mode == "blackBackWhiteFront":
        binaryArr = (arr > thresh).astype(np.uint8)
    elif mode == "dynamic":
        if isBlackOnWhite(arr) == True:
            binaryArr = (arr < thresh).astype(np.uint8)
        else:
            binaryArr = (arr > thresh).astype(np.uint8)
    else:
        binaryArr = (arr < thresh).astype(np.uint8)
        
    yLen, xLen = binaryArr.shape
    
    return binaryArr, xLen, yLen

def findContours(binaryArr, xLen, yLen, retrievalMode, overWriteMode, overWriteLen):
    yContourArr = np.array([], dtype=int)
    xContourArr = np.array([], dtype=int)
    blankIndexArr = np.array([], dtype=int)

    #computes and stores contours in a tuple
    contours, _ = cv2.findContours(binaryArr, retrievalMode, cv2.CHAIN_APPROX_NONE)
    
    #appends contours consecutively to the axis' buffer
    #overWriteMode appends the first overWriteLen*2 data points of a contour to its end
    #it gives more time to the optocoupler to properly blank/unblank the CRT and as such reduces warping and intercontour strokes at the expense of larger data buffers
    if overWriteMode == True:
        for i in range(len(contours)):
            yContourArrBuff = np.transpose(np.array(contours[i]))[1][0]
            xContourArrBuff = np.transpose(np.array(contours[i]))[0][0]

            if len(contours[i]) > overWriteLen:
                yContourArrBuff = np.append(yContourArrBuff, yContourArrBuff[:overWriteLen*2])
                xContourArrBuff = np.append(xContourArrBuff, xContourArrBuff[:overWriteLen*2])
                
            yContourArr = np.append(yContourArr, yContourArrBuff)
            xContourArr = np.append(xContourArr, xContourArrBuff)

            blankIndexArrBuff = np.zeros(len(xContourArrBuff), dtype=int)

            if len(xContourArr) > overWriteLen:
                blankIndexArrBuff[:overWriteLen] = 1
                blankIndexArrBuff[-overWriteLen:] = 1
            else:
                blankIndexArrBuff = np.ones(len(xContourArrBuff), dtype=int)
            
            blankIndexArr = np.append(blankIndexArr, blankIndexArrBuff)
    else:
        for i in range(len(contours)):
            yContourArrBuff = np.transpose(np.array(contours[i]))[1][0]
            yContourArr = np.append(yContourArr, yContourArrBuff)

            xContourArrBuff = np.transpose(np.array(contours[i]))[0][0]
            xContourArr = np.append(xContourArr, xContourArrBuff)

            blankIndexArrBuff = np.zeros(len(contours[i]), dtype=int)

            if len(contours[i]) > 2:
                blankIndexArrBuff[:1] = 1
                blankIndexArrBuff[-1:] = 1
            else:
                blankIndexArrBuff = np.ones(len(contours[i]), dtype=int)
            
            blankIndexArr = np.append(blankIndexArr, blankIndexArrBuff)
    
    #scales the data to a 8bit basis
    sx = 255/(xLen+len(contours))
    sy = 255/(yLen+len(contours))

    #computes and returns final data
    blankArr = blankIndexArr
    xArr = (sx*xContourArr).astype(int)
    yArr = (255 - sy*yContourArr).astype(int)
    dataLen = len(xArr)
    
    return dataLen, blankArr, xArr, yArr


def transmitStaticImage(register, dataLen, blankArr, xArr, yArr):
    #sends register to write to
    ser.write(register.encode())
    ser.read(1) #waits for ACK

    #sends data length (1-1024)
    data = dataLen.to_bytes(2, byteorder='big')
    ser.write(data)
    ser.read(1) #waits for ACK

    #sends blanking array
    data = b''.join(int(v).to_bytes(1, byteorder='big') for v in blankArr)
    ser.write(data)
    ser.read(1) #waits for ACK

    #sends X coordinates array
    data = b''.join(int(v).to_bytes(1, byteorder='big') for v in xArr)
    ser.write(data)
    ser.read(1) #waits for ACK

    #sends Y coordinates array
    data = b''.join(int(v).to_bytes(1, byteorder='big') for v in yArr)
    ser.write(data)
    ser.read(1) #waits for ACK

## Static image example

In [5]:
binaryArr, xLen, yLen = getImageFromFile("bb.png", 1, 128, mode="dynamic")
dataLen, blankArr, xArr, yArr = findContours(binaryArr, xLen, yLen, cv2.RETR_LIST, overWriteMode = True, overWriteLen = 1)

if dataLen <= 1024:
    ser = serial.Serial('COM11', 921600, timeout=1)
    time.sleep(0.05)

    transmitStaticImage("w0", dataLen, blankArr, xArr, yArr)

    ser.close()
    print(f"Data length : {dataLen}")
else:
    print(f"Data length too long : {dataLen}>1024")

Data length : 976


## Streaming example

In [11]:
ser = serial.Serial('COM11', 921600, timeout=5)
time.sleep(0.05)

iterationN = 300

for i in range(iterationN):
    binaryArr, xLen, yLen = getImage("windowname", 0.2, 128, mode="dynamic", customWindowSize = True, bounds=[200,1200,400,1500])
    dataLen, blankArr, xArr, yArr = findContours(binaryArr, xLen, yLen, cv2.RETR_LIST, overWriteMode = False, overWriteLen = 50)
    
    if(dataLen > 950):
        decimationFactor = np.round(950/dataLen, 1)
        blankArrBuff = decimate(blankArr, decimationFactor)
        xArrBuff = decimate(xArr, decimationFactor)
        yArrBuff = decimate(yArr, decimationFactor)
        dataLenBuff = len(xArrBuff)
    else:
        dataLenBuff = dataLen
        blankArrBuff = blankArr
        xArrBuff = xArr
        yArrBuff = yArr
        
    if(dataLenBuff<1024 and dataLenBuff>0):
        transmitStaticImage("w0", dataLenBuff, blankArrBuff, xArrBuff, yArrBuff)
    else:
        pass

ser.close()

## Flight data application example

In [ ]:
#sets up request to API
url = "https://opensky-network.org/api/states/all"

lonIndex = 5
latIndex = 6
onGroundIndex = 8
angIndex = 10

params = {
    "lamin": 43.36,
    "lamax": 43.83,
    "lomin": 0.995,
    "lomax": 1.862
}

iterationN = 300

for i in range(iterationN):
    binaryArr, xLen, yLen = getImageFromFile("flightradar/tls.png", 1, 128, mode="dynamic")
    dataLen, blankArr, xArr, yArr = findContours(binaryArr, xLen, yLen, cv2.RETR_LIST, overWriteMode = True, overWriteLen = 50)
    
    #sends requests to API
    response = requests.get(url, params=params)
    data = response.json()
    flightsN = len(list(data.values())[1])

    #gets map image
    mapImg = Image.open("flightradar/blanktls.png").convert("L")
    mapW, mapH = mapImg.size
    sx = mapW/(params["lomax"]-params["lomin"])
    sy = mapH/(params["lamax"]-params["lamin"])

    #gets plane image
    planeImg = Image.open("flightradar/plane0.png").convert("L")
    w, h = planeImg.size
    planeImg = planeImg.resize((int(w*0.8), int(h*0.8)))
    basePlaneImg = planeImg

    #applies planes images
    for i in range(flightsN):
        if list(data.values())[1][i][onGroundIndex] == False:
            planeImg = basePlaneImg.rotate(-list(data.values())[1][i][angIndex], expand=0, fillcolor=255)
            mapImg.paste(planeImg, (int(sx*(list(data.values())[1][i][lonIndex]-params["lomin"])),int(mapH-sy*(list(data.values())[1][i][latIndex]-params["lamin"]))))

    binaryArrPlanes = (np.array(mapImg) < 128).astype(np.uint8)
    yLenPlanes, xLenPlanes = binaryArr.shape

    dataLenPlanes, blankArrPlanes, xArrPlanes, yArrPlanes = findContours(binaryArrPlanes, xLenPlanes, yLenPlanes, cv2.RETR_LIST, overWriteMode = True, overWriteLen = 1)

    xArr = np.append(xArr, xArrPlanes)
    yArr = np.append(yArr, yArrPlanes)
    blankArr = np.append(blankArr, blankArrPlanes)
    dataLen = dataLen + dataLenPlanes

    ser = serial.Serial('COM11', 921600, timeout=1)
    time.sleep(0.05)

    if(dataLen > 950):
        decimationFactor = np.round(950/dataLen, 1)
        blankArrBuff = decimate(blankArr, decimationFactor)
        xArrBuff = decimate(xArr, decimationFactor)
        yArrBuff = decimate(yArr, decimationFactor)
        dataLenBuff = len(xArrBuff)
    else:
        dataLenBuff = dataLen
        blankArrBuff = blankArr
        xArrBuff = xArr
        yArrBuff = yArr

    if(dataLenBuff<1024 and dataLenBuff>0):
        transmitStaticImage("w0", dataLenBuff, blankArrBuff, xArrBuff, yArrBuff)
    else:
        pass

    ser.close()
    time.sleep(5)